# Sub-THz Blockage Prediction — LoRA Fine-tuning (Colab **or** Kaggle)

Trains the two LoRA architectures (Arch A forecaster, Arch B classifier) on
**TimesFM 2.5** and reproduces the paper's metrics + figures on a cloud GPU.
The `src/` code is identical to `main`; this notebook only orchestrates it
(platform + GPU-architecture detection, output persistence, and **resume-safe**
experiment loops that survive disconnects).

### Before running
* **Colab:** `Runtime → Change runtime type → GPU`.
* **Kaggle:** right panel → `Settings → Accelerator → GPU P100` **and**
  `Settings → Internet → On` (needed for pip / git clone / model download).

Then run the cells top to bottom.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — enable a GPU accelerator first.'
p = torch.cuda.get_device_properties(0)
print(f'{p.name}  |  cc {p.major}.{p.minor}  |  {p.total_memory/1e9:.1f} GB  |  torch {torch.__version__}')

## 2. Clone the repo (code + data) and install deps

The dataset is committed in the repo, so a shallow clone brings everything.
(**Kaggle:** make sure Internet is **On**, or this cell fails.) For a private
repo, paste a GitHub token; leave blank if public.

> **Ignore the `RAPIDS` / `cu12` dependency-conflict warnings** pip prints
> (`libraft`, `librmm`, `rmm`, `ucxx`, `rapids-dask-*`, `nvforest`, ...). Those
> are pre-existing version skew in Kaggle's base image and have nothing to do
> with this install — we don't use RAPIDS. The check at the end of the cell is
> the one that matters.

In [ ]:
REPO = 'github.com/kaefcatcher/THz_blockage.git'
BRANCH = 'collab'  # the branch holding this notebook
TOKEN = ''  # e.g. 'ghp_...'; leave '' for a public repo

import os, sys
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.isdir('THz_blockage'):
    !git clone --depth 1 --branch {BRANCH} {url}
%cd THz_blockage
# torch is preinstalled on Colab/Kaggle; TimesFM 2.5 needs transformers >= 5.12.
# (Any RAPIDS/cu12 conflict warnings pip prints are pre-existing image skew —
# harmless here; only the import check below matters.)
!pip install -q -U 'transformers>=5.12' 'peft>=0.13' 'safetensors>=0.4' einops

# --- the only check that matters: do OUR packages import at the right version? ---
import importlib, transformers, peft
importlib.reload(transformers)
try:
    from transformers import TimesFm2_5ModelForPrediction  # needs transformers >= 5.12
    print(f'\u2713 ready: transformers {transformers.__version__} | peft {peft.__version__} | TimesFM 2.5 OK')
except ImportError as e:
    print('transformers in THIS kernel is too old:', transformers.__version__)
    print('Fix: Run -> Restart session (or Restart kernel), then re-run this cell.')
    raise

## 3. Output persistence (platform-aware)

* **Kaggle:** outputs live under `/kaggle/working` (this clone) — kept across
  cell re-runs, and across sessions when you click **Save Version** or push to
  GitHub (last cell).
* **Colab:** optionally mount Drive and symlink `outputs/` into it so
  checkpoints/CSVs survive disconnects.

In [ ]:
import os, sys
IS_KAGGLE = os.path.isdir('/kaggle')
IS_COLAB = os.path.isdir('/content') and not IS_KAGGLE
os.makedirs('outputs/results', exist_ok=True)
if IS_KAGGLE:
    print('Kaggle: outputs under /kaggle/working — Save Version (or push) to keep across sessions.')
elif IS_COLAB:
    USE_DRIVE = True  # False -> keep outputs only in the ephemeral VM
    if USE_DRIVE:
        from google.colab import drive; drive.mount('/content/drive')
        DRIVE_OUT = '/content/drive/MyDrive/thz_blockage_outputs'; os.makedirs(DRIVE_OUT, exist_ok=True)
        if os.path.islink('outputs') or not os.listdir('outputs'):
            if os.path.islink('outputs'): os.unlink('outputs')
            elif os.path.isdir('outputs'): os.rmdir('outputs')
            os.symlink(DRIVE_OUT, 'outputs')
        print('Colab: outputs ->', os.path.realpath('outputs'))
else:
    print('Generic env: outputs in ./outputs')

## 4. Verify the data pipeline and LoRA setup (Steps 1–2)

First model load downloads the TimesFM 2.5 weights (~0.9 GB) into the cache.

In [ ]:
!python src/data.py
!python src/lora_common.py

## 5. GPU-architecture-aware memory flags

**bfloat16 needs Ampere (compute capability 8.0+).** P100 (6.0), T4 (7.5) and
V100 (7.0) have no native bf16, so on those we use **fp32 + per-layer gradient
checkpointing** — reliable and a comfortable fit in 16 GB. Ampere GPUs use bf16.
None of this changes results (accumulation preserves the effective batch).

In [ ]:
import torch
name = torch.cuda.get_device_name(0)
major = torch.cuda.get_device_capability()[0]
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = major >= 8  # Ampere+
if bf16_ok and gb >= 24:        # A100 / L4 / big Ampere
    CLI = '--bf16 --batch-size 64'
    TRAIN_KWARGS = dict(dtype='bf16', batch_size=64)
elif bf16_ok:                   # smaller Ampere
    CLI = '--bf16 --grad-checkpoint --batch-size 16 --accum-steps 4'
    TRAIN_KWARGS = dict(dtype='bf16', grad_checkpoint=True, batch_size=16, accum_steps=4)
else:                           # P100 / T4 / V100 -> fp32
    CLI = '--grad-checkpoint --batch-size 32 --accum-steps 2'
    TRAIN_KWARGS = dict(grad_checkpoint=True, batch_size=32, accum_steps=2)
print(f'{name}  cc{major}.x  {gb:.0f}GB  bf16={bf16_ok}  ->  {CLI}')

## 6. Train the canonical Arch A + Arch B checkpoints (N=216)

Watch the printed `peak=X.XXGB`. If it OOMs, append `--low-mem` or lower
`--batch-size`. Each is a full 30-epoch run with early stopping.

In [ ]:
!python src/lora_arch_a.py {CLI}

In [ ]:
!python src/lora_arch_b.py --loss bce {CLI}
!python src/lora_arch_b.py --loss focal {CLI}

## 7. Main metrics table (Step 5)

In [ ]:
import sys; sys.path.insert(0, 'src')
import experiments
experiments.build_metrics_main(train_kwargs=TRAIN_KWARGS)

## 8. Data-efficiency experiment — **resume-safe** (Step 6)

42 runs (7 sizes x 3 seeds x 2 archs) — the long pole, longer than one free
session. The loop **skips runs already in the CSV** and appends incrementally,
so re-run this cell in a new session to continue (persist `outputs/` first —
Save Version on Kaggle, Drive on Colab, or push to GitHub). For a quick first
pass, set `SEEDS = [0]` and/or trim `N_GRID`.

In [ ]:
import os, pandas as pd, sys
sys.path.insert(0, 'src')
import data, importlib, experiments; importlib.reload(experiments)
from experiments import run_one, N_GRID, SEEDS, TW

EFF = 'outputs/results/metrics_data_efficiency.csv'; os.makedirs('outputs/results', exist_ok=True)
done = set()
if os.path.exists(EFF):
    _d = pd.read_csv(EFF)
    done = {tuple(r) for r in _d[['arch','N_traces','seed']].itertuples(index=False, name=None)}

pool = data.get_train_pool_files()
for N in N_GRID:
    for seed in SEEDS:
        files = data.sample_traces_stratified(pool, N, seed=seed)
        for arch in ('A', 'B'):
            if (arch, N, seed) in done:
                print(f'skip arch={arch} N={N} seed={seed} (done)'); continue
            m = run_one(arch, files, seed=seed, loss='bce', train_kwargs=TRAIN_KWARGS)
            row = {'arch': arch, 'N_traces': N, 'seed': seed, 'Tw_ms': TW,
                   'acc': round(m['acc'],4), 'precision': round(m['precision'],4),
                   'recall': round(m['recall'],4), 'f1': round(m['f1'],4)}
            pd.DataFrame([row]).to_csv(EFF, mode='a', header=not os.path.exists(EFF), index=False)
            print(f"done arch={arch} N={N} seed={seed} f1={m['f1']:.4f}")
print('data-efficiency complete')

## 9. Diversity experiment — **resume-safe** (Step 7)

In [ ]:
import os, pandas as pd, sys, tempfile, pathlib
sys.path.insert(0, 'src')
import data, evaluate as ev, lora_arch_b
from experiments import _sample_config, _div_row, report_diversity, TW

DIV = 'outputs/results/metrics_diversity.csv'
pool = data.get_train_pool_files(); eval_files = data.get_eval_files()
theta = data.get_theta(); cfgs = sorted({data.meta_of(p)['set'] for p in eval_files})
done = set()
if os.path.exists(DIV):
    _d = pd.read_csv(DIV)
    done = {tuple(r) for r in _d[['condition','seed']].itertuples(index=False, name=None)}

for seed in [0, 1, 2]:
    conds = {'balanced': data.sample_traces_stratified(pool, 80, seed=seed),
             'homogeneous': _sample_config(pool, 1, 40, seed) + _sample_config(pool, 2, 40, seed)}
    for cond, files in conds.items():
        if (cond, seed) in done:
            print(f'skip {cond} seed={seed} (done)'); continue
        with tempfile.TemporaryDirectory() as tmp:
            model, _f1, _t = lora_arch_b.train(loss_kind='bce', train_files=files, seed=seed,
                                               save_dir=pathlib.Path(tmp), enforce_gate=False, **TRAIN_KWARGS)
            rows = [_div_row(cond, seed, 'overall', ev.evaluate(model, eval_files, TW, theta, 'arch_b'))]
            for c in cfgs:
                ef = data.files_for_configs(eval_files, [c])
                rows.append(_div_row(cond, seed, str(c), ev.evaluate(model, ef, TW, theta, 'arch_b')))
        pd.DataFrame(rows).to_csv(DIV, mode='a', header=not os.path.exists(DIV), index=False)
        print(f'done {cond} seed={seed}'); del model
report_diversity(pd.read_csv(DIV))

## 10. Figures (Step 8)

In [ ]:
!python src/figures.py --with-models
for f in ('data_efficiency_curve', 'pr_curve', 'metrics_table'):
    print('outputs/figures/' + f + '.pdf')

## 11. Save / export results

Outputs are a few MB (tiny LoRA adapters + CSVs + PDFs). Zip + download, or
push back to the branch — pushing is also how you **resume across sessions**:
next session's clone restores the partial CSVs and the loops continue.

In [ ]:
import os, shutil
IS_KAGGLE = os.path.isdir('/kaggle')
dest = '/kaggle/working/thz_outputs' if IS_KAGGLE else '/tmp/thz_outputs'
shutil.make_archive(dest, 'zip', 'outputs'); print('zip ->', dest + '.zip')
if os.path.isdir('/content') and not IS_KAGGLE:
    from google.colab import files; files.download('/tmp/thz_outputs.zip')
elif IS_KAGGLE:
    print('Kaggle: grab it from the right-side Output panel, or click Save Version.')

# --- resume across sessions: push partial results to the branch ---
# !git config user.email 'you@example.com' && git config user.name 'you'
# !git add -f outputs && git commit -m 'cloud: progress' && git push https://{TOKEN}@{REPO} HEAD:{BRANCH}